In [0]:
from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F

dbutils.widgets.text("ingest_date", "2026-11-06")
ingest_date = dbutils.widgets.get("ingest_date")

BRONZE = "workspace.bank_lakehouse.bronze_osm_banks"
RULES = "workspace.bank_lakehouse.ref_bank_rules"
SILVER = "workspace.bank_lakehouse.silver_osm_banks"
QUARANTINE = "workspace.bank_lakehouse.quarantine_osm_banks"


def normalize(col):
    text = F.lower(col)
    text = F.translate(text, "éèêëàâäîïôöùûüç", "eeeeaaaiioouuuc")
    text = F.regexp_replace(text, r"[^\p{L}\p{N}]+", " ")
    return F.trim(text)


# 1. Lire bronze et préparer les colonnes
src = (
    spark.table(BRONZE)
    .filter(F.col("_ingest_date") == ingest_date)
    .select(
        F.concat_ws("/", "osm_type", F.col("osm_id").cast("string")).alias("osm_key"),
        "osm_type",
        "osm_id",
        F.coalesce("lat", "center_lat").alias("lat"),
        F.coalesce("lon", "center_lon").alias("lon"),
        F.col("tags")["name"].alias("name"),
        F.col("tags")["brand"].alias("brand"),
        F.col("tags")["operator"].alias("operator"),
        F.col("tags")["addr:city"].alias("addr_city"),
        "_osm_snapshot_at",
    )
    .withColumn("match_text", normalize(F.concat_ws(" ", "brand", "name", "operator")))
)
n_src = src.count()
if n_src < 1000:
    raise ValueError(
        f"Seulement {n_src} agences en bronze pour {ingest_date} : MERGE annulé "
        "pour ne pas désactiver le réseau entier."
    )
print("Agences en entrée :", n_src)
# 2. Appliquer les règles de normalisation des banques
by_key = Window.partitionBy("osm_key")
matched = (
    src.join(F.broadcast(spark.table(RULES)), F.expr("match_text rlike pattern"), "left")
    .withColumn("rn", F.row_number().over(
        by_key.orderBy(F.col("priority").asc_nulls_last(), F.col("bank"))
    ))
    .filter("rn = 1")
    .select(
        *[c for c in src.columns if c != "match_text"],
        F.coalesce("bank", F.lit("Inconnue")).alias("bank_name"),
        F.coalesce("institution_type", F.lit("non identifié")).alias("institution_type"),
    )
)
# 2b. Rattacher chaque agence à sa région (jointure spatiale)
regions = (
    spark.table("workspace.bank_lakehouse.bronze_regions")
    .select(
        F.col("shape_iso").alias("region_iso"),
        F.col("shape_name").alias("region_name"),
        F.expr("st_geomfromgeojson(geometry_geojson)").alias("region_geom"),
    )
)

matched = (
    matched
    .join(F.broadcast(regions), F.expr("st_contains(region_geom, st_point(lon, lat, 4326))"), "left")
    .withColumn("rn", F.row_number().over(by_key.orderBy(F.col("region_iso").asc_nulls_last())))
    .filter("rn = 1")
    .drop("rn", "region_geom")
)
# 3. Contrôles qualité et quarantaine
checked = matched.withColumn(
    "dq_error",
    F.when(F.col("lat").isNull() | F.col("lon").isNull(), "coordonnées manquantes")
     .when(~F.col("lat").between(20.5, 36.5) | ~F.col("lon").between(-17.5, -0.5), "hors zone attendue"),
)

invalid = checked.filter("dq_error is not null") \
    .withColumn("_ingest_date", F.lit(ingest_date).cast("date"))
q_writer = invalid.write.format("delta").partitionBy("_ingest_date").option("mergeSchema", "true")
if spark.catalog.tableExists(QUARANTINE):
    q_writer.mode("overwrite").option("replaceWhere", f"_ingest_date = '{ingest_date}'").saveAsTable(QUARANTINE)
else:
    q_writer.mode("overwrite").saveAsTable(QUARANTINE)

# 4. Empreinte des colonnes métier pour détecter les changements
business_cols = ["lat", "lon", "name", "brand", "operator", "addr_city",
                 "bank_name", "institution_type", "region_iso", "region_name"]
valid = (
    checked.filter("dq_error is null").drop("dq_error")
    .withColumn("row_hash", F.sha2(F.concat_ws("||", *[
        F.coalesce(F.col(c).cast("string"), F.lit("<null>")) for c in business_cols
    ]), 256))
)

# 5. Créer la table silver vide si elle n'existe pas
if not spark.catalog.tableExists(SILVER):
    (
        valid.limit(0)
        .withColumn("first_seen_date", F.lit(None).cast("date"))
        .withColumn("is_active", F.lit(None).cast("boolean"))
        .withColumn("updated_at", F.lit(None).cast("timestamp"))
        .write.format("delta").saveAsTable(SILVER)
    )

# 6. MERGE : appliquer seulement les changements
def apply_merge(target_table, source_df, run_date):
    change_cols = {c: f"s.{c}" for c in business_cols + ["row_hash", "_osm_snapshot_at"]}
    insert_cols = {c: f"s.{c}" for c in source_df.columns}
    (
        DeltaTable.forName(spark, target_table).alias("t")
        .merge(source_df.alias("s"), "t.osm_key = s.osm_key")
        .whenMatchedUpdate(
            condition="t.row_hash <> s.row_hash OR NOT t.is_active",
            set={**change_cols, "is_active": "true", "updated_at": "current_timestamp()"},
        )
        .whenNotMatchedInsert(values={
            **insert_cols,
            "first_seen_date": f"DATE'{run_date}'",
            "is_active": "true",
            "updated_at": "current_timestamp()",
        })
        .whenNotMatchedBySourceUpdate(
            condition="t.is_active",
            set={"is_active": "false", "updated_at": "current_timestamp()"},
        )
        .execute()
    )


apply_merge(SILVER, valid, ingest_date)

print("Silver à jour pour", ingest_date)

In [0]:
silver = spark.table(SILVER)

silver.agg(
    F.count("*").alias("lignes"),
    F.countDistinct("osm_key").alias("cles_distinctes"),
    F.sum(F.col("is_active").cast("int")).alias("actives"),
).show()

print("Quarantaine :", spark.table(QUARANTINE).filter(F.col("_ingest_date") == ingest_date).count())

display(silver.groupBy("institution_type").count().orderBy(F.desc("count")))

In [0]:
display(
    spark.sql(f"DESCRIBE HISTORY {SILVER}")
    .select(
        "version", "operation",
        F.col("operationMetrics")["numTargetRowsInserted"].alias("inserees"),
        F.col("operationMetrics")["numTargetRowsUpdated"].alias("modifiees"),
    )
    .orderBy("version")
)